In [1]:
# imports

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import numpy as np
from pathlib import Path
import cv2

In [2]:
def compress_image(img, quality=50):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, enc = cv2.imencode('.jpg', img, encode_param)
    img = cv2.imdecode(enc, 1)
    return img

def augment_frame(img):
    if np.random.rand() < 0.5:
        img = compress_image(img, quality=np.random.randint(30, 70))

    if np.random.rand() < 0.3:
        img = cv2.GaussianBlur(img, (5,5), 0)

    if np.random.rand() < 0.3:
        noise = np.random.normal(0, 10, img.shape)
        img = np.clip(img + noise, 0, 255).astype(np.uint8)

    if np.random.rand() < 0.3:
        alpha = np.random.uniform(0.8, 1.2)
        beta = np.random.uniform(-20, 20)
        img = np.clip(alpha * img + beta, 0, 255).astype(np.uint8)

    if np.random.rand() < 0.3:
        img = cv2.flip(img, 1)

    return img

class DeepfakeDataset(Dataset):
    
    def __init__(self, folder, mode = "train"):
        self.files = sorted(Path(folder).glob("*.pt"))
        self.mode = mode

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = torch.load(self.files[idx])

        spatial = data["spatial"]
        frequency = data["frequency"]
        label = data["label"]

        if self.mode == "train":
            spatial = [augument_frame(f) for f in spatial]

        return {
            "spatial": spatial,
            "frequency": frequency,
            "label": label
        }

# train_dataset = DeepfakeDataset("preprocessed/train", mode="train")

# sample = train_dataset[0]

# print("Training Dataset")
# print(sample["spatial"].shape)
# print(sample["frequency"].shape)
# print(sample["label"])

# plt.imshow(sample["spatial"][0].permute(1, 2, 0))
# plt.title("Spatial")
# plt.axis("off")
# plt.show()

# plt.imshow(sample["frequency"][0][0])
# plt.title("Frequency")
# plt.axis("off")
# plt.show()

In [3]:
print("Validation Dataset")
val_dataset = DeepfakeDataset("preprocessed/val", mode="val")
sample = val_dataset[0]
print(sample["spatial"].shape)
print(sample["frequency"].shape)
print(sample["label"])
print("\n\nTest Dataset")
test_dataset = DeepfakeDataset("preprocessed/test", mode="test")
sample = test_dataset[0]
print(sample["spatial"].shape)
print(sample["frequency"].shape)
print(sample["label"])

Validation Dataset
torch.Size([10, 3, 224, 224])
torch.Size([10, 1, 224, 224])
tensor(1.)


Test Dataset
torch.Size([10, 3, 224, 224])
torch.Size([10, 1, 224, 224])
tensor(1.)


In [4]:
# train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [12]:
class Model0(nn.Module):
    def __init__(self):
        super().__init__()

        self.spatial_net = models.mobilenet_v2(weights="IMAGENET1K_V1").features
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        self.freq_net = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding = 1),
            nn.ReLU(),
            nn.Conv2d(8, 16, 3, padding = 1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1280 + 16, 1)
        )

    def forward(self, spatial, frequency):
        B, T, C, H, W = spatial.shape

        spatial = spatial.view(B * T, C, H, W)
        frequency = frequency.view(B * T, 1, H, W)

        s_feat = self.spatial_net(spatial)
        s_feat = self.pool(s_feat)
        f_feat = self.freq_net(frequency)

        s_feat = s_feat.view(B, T, -1)
        f_feat = f_feat.view(B, T, -1)

        s_feat = s_feat.mean(dim=1)
        f_feat = f_feat.mean(dim=1)

        combined = torch.cat([s_feat, f_feat], dim = 1)
        out = self.fc(combined)

        return out


model = Model0()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

In [13]:
def train(model, epochs, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    
    for batch in loader:
        spatial = batch["spatial"].float()
        frequency = batch["frequency"].float()
        labels = batch["label"].unsqueeze(1).float()
    
        outputs = model(spatial, frequency)
        loss = criterion(outputs, labels)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        acc = (preds == labels).float().mean()
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        total_loss += loss.item()

    return loss/len(loader)



0.7225808501243591
0.6733680963516235
0.6445711255073547
0.6110744476318359
0.573067307472229
0.5440714955329895
0.5146624445915222
0.47509875893592834
0.44736406207084656
0.42208269238471985
0.39639121294021606
0.36149561405181885
0.3782897889614105
0.32359611988067627
0.2992696762084961
0.28133660554885864
0.27956831455230713
0.25602588057518005
0.2390885055065155
0.22310511767864227
0.1991320252418518
0.18661467730998993
0.17427261173725128
0.15890204906463623
0.15544214844703674
0.14189600944519043
0.1269056648015976
0.12508049607276917
0.11537860333919525
0.10808587074279785
0.1284821778535843
0.12745574116706848


KeyboardInterrupt: 

In [ ]:
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            spatial = batch["spatial"].float()
            frequency = batch["frequency"].float()
            labels = batch["label"].unsqueeze(1).float()

            outputs = model(spatial, frequency)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)